<a href="https://colab.research.google.com/github/meizhong986/WhisperJAV/blob/main/notebook/WhisperJAV_colab_edition_expert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WhisperJAV Colab Edition v1.9.5 (Expert)

Run Step 1 to configure. Then either put videos in `Google Drive/WhisperJAV/` yourself or paste
links into Step 2 — the notebook downloads them and converts the sound to MP3. Run Step 3 to
transcribe, and — if you asked for an AI translation — Step 4. The session disconnects itself
when the work is finished, so no GPU credits are spent waiting.

## Pipelines

| Pipeline | What it is |
|---|---|
| **balanced** | faster-whisper with its own built-in speech detector. The default. |
| **fidelity** | Slowest and most thorough; uses FireRedVAD to find speech. |
| **fast** / **faster** | Quicker, less thorough. |
| **transformers** | Hugging Face Whisper models, including the Kotoba Japanese models. |
| **qwen** | Qwen3-ASR, or anime-whisper for anime and visual-novel dialogue. |

## The audio settings

| Setting | What it controls |
|---|---|
| **Built-in VAD** | Which Silero build the balanced pipeline's internal detector uses (3.1, 4.0, 6.2). |
| **Speech segmenter** | The separate speech detector used by every pipeline except balanced: whisperseg, firered-vad, silero, nemo, whisper-vad, ten. |
| **Scene detection** | How the audio is cut into pieces before recognition: semantic, auditok, silero. |
| **Audio clean-up** | Noise and level treatment for poor sources: ffmpeg-dsp, clearvoice, zipenhancer, bs-roformer. |

**Balanced and the speech segmenter.** The balanced pipeline uses faster-whisper's own detector and
does not accept a separate segmenter. Choose the Silero build under *built-in VAD* instead, or pick
another pipeline for that pass.

**Audio clean-up** applies to a two-pass run and to the qwen pipeline. On a single balanced,
fidelity, fast, faster or transformers pass it has no effect, and Step 1 says so.

## Downloading from links

Step 2 is optional and needs no GPU. Paste any links — streaming pages or plain file URLs —
and it fetches them with [yt-dlp](https://github.com/yt-dlp/yt-dlp), converts the sound to MP3
and saves the result in the Drive folder. The videos themselves are not kept. Links already
fetched on an earlier run are skipped automatically; delete `.download-archive.txt` in the
folder if you ever want a fresh fetch. Step 3 then transcribes those MP3s along with any files
you put in the folder yourself.

## Two passes

Turning on *two passes* runs both passes and merges the result — slower, usually more accurate.
Each pass has its own pipeline, model and audio settings.

## Translation

**local** runs a translation model on this session's GPU and needs no key. **ollama** and **custom**
talk to an OpenAI-compatible server whose address you give. The rest are cloud services and need an
API key.

---
<sub>If you find WhisperJAV useful, consider supporting the project: <a href="https://buymeacoffee.com/meizhong">buymeacoffee.com/meizhong</a></sub>

In [2]:
#@title Step 1: Expert Configuration { display-mode: "form" }

#@markdown ## Files and output
folder_name = "WhisperJAV" #@param {type:"string"}
subtitle_language = "English (AI translate)" #@param ["Japanese", "English (Whisper direct)", "English (AI translate)"]

#@markdown ---
#@markdown ## 1. First pass
pass1_pipeline = "balanced" #@param ["balanced", "fidelity", "fast", "faster", "transformers", "qwen"]
pass1_sensitivity = "aggressive" #@param ["conservative", "balanced", "aggressive"]

#@markdown **Model.** The box that applies is the one for the pipeline you chose above:
#@markdown *whisper model* for balanced / fidelity / fast / faster, *transformers model* for transformers,
#@markdown *qwen generator* for qwen. The other two boxes are ignored.
pass1_whisper_model = "automatic" #@param ["automatic", "large-v2", "large-v3", "turbo"]
pass1_transformers_model = "kotoba-bilingual-v1.0" #@param ["kotoba-bilingual-v1.0", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"]
pass1_qwen_generator = "qwen3" #@param ["qwen3", "anime-whisper"]

#@markdown **Speech detection.** Balanced runs faster-whisper's own detector, and *built-in VAD* picks which
#@markdown Silero build it uses. Every other pipeline runs a separate detector, chosen in *speech segmenter*.
pass1_builtin_vad = "automatic" #@param ["automatic", "3.1", "4.0", "6.2"]
pass1_speech_segmenter = "automatic" #@param ["automatic", "whisperseg", "firered-vad", "silero-v3.1", "silero-v4.0", "silero-v6.2", "nemo", "nemo-lite", "whisper-vad", "ten", "none"]

#@markdown **Scene detection**
pass1_scene_detector = "automatic" #@param ["automatic", "semantic", "auditok", "silero", "none"]

#@markdown **Audio clean-up** (two-pass runs and the qwen pipeline only).
#@markdown <font size="1">htdemucs and bs-roformer separate the voice from music and effects;
#@markdown zipenhancer and clearvoice reduce noise; ffmpeg-dsp applies the filters below.
#@markdown htdemucs brings an 84 MB model, fetched once from Meta's site (not Hugging Face,
#@markdown so a Hugging Face mirror does not redirect it).
#@markdown on first use.</font>
pass1_speech_enhancer = "none" #@param ["none", "ffmpeg-dsp", "clearvoice", "zipenhancer", "bs-roformer", "htdemucs"]
#@markdown <font size="1">The filters below apply only when the clean-up above is ffmpeg-dsp.</font>
pass1_ffmpeg_loudnorm = True #@param {type:"boolean"}
pass1_ffmpeg_normalize = False #@param {type:"boolean"}
pass1_ffmpeg_compress = False #@param {type:"boolean"}
pass1_ffmpeg_denoise = False #@param {type:"boolean"}
pass1_ffmpeg_highpass = False #@param {type:"boolean"}
pass1_ffmpeg_lowpass = False #@param {type:"boolean"}
pass1_ffmpeg_deess = False #@param {type:"boolean"}
pass1_ffmpeg_amplify = False #@param {type:"boolean"}

#@markdown ---
#@markdown ## Two passes
#@markdown Leave this off for a single pass. With it on, both passes run and their subtitles are merged.
use_two_step = False #@param {type:"boolean"}
merge_method = "prefer first pass" #@param ["automatic", "keep all", "prefer first pass", "prefer second pass", "first pass on overlap", "second pass on overlap", "longest line"]

#@markdown ---
#@markdown ## 2. Second pass *(only used when two passes is on)*
pass2_pipeline = "qwen" #@param ["balanced", "fidelity", "fast", "faster", "transformers", "qwen"]
pass2_sensitivity = "aggressive" #@param ["conservative", "balanced", "aggressive"]

pass2_whisper_model = "automatic" #@param ["automatic", "large-v2", "large-v3", "turbo"]
pass2_transformers_model = "kotoba-bilingual-v1.0" #@param ["kotoba-bilingual-v1.0", "kotoba-v2.0", "kotoba-v2.1", "kotoba-v2.2"]
pass2_qwen_generator = "qwen3" #@param ["qwen3", "anime-whisper"]

pass2_builtin_vad = "automatic" #@param ["automatic", "3.1", "4.0", "6.2"]
pass2_speech_segmenter = "automatic" #@param ["automatic", "whisperseg", "firered-vad", "silero-v3.1", "silero-v4.0", "silero-v6.2", "nemo", "nemo-lite", "whisper-vad", "ten", "none"]

pass2_scene_detector = "automatic" #@param ["automatic", "semantic", "auditok", "silero", "none"]

pass2_speech_enhancer = "none" #@param ["none", "ffmpeg-dsp", "clearvoice", "zipenhancer", "bs-roformer", "htdemucs"]
#@markdown <font size="1">The filters below apply only when the clean-up above is ffmpeg-dsp.</font>
pass2_ffmpeg_loudnorm = True #@param {type:"boolean"}
pass2_ffmpeg_normalize = False #@param {type:"boolean"}
pass2_ffmpeg_compress = False #@param {type:"boolean"}
pass2_ffmpeg_denoise = False #@param {type:"boolean"}
pass2_ffmpeg_highpass = False #@param {type:"boolean"}
pass2_ffmpeg_lowpass = False #@param {type:"boolean"}
pass2_ffmpeg_deess = False #@param {type:"boolean"}
pass2_ffmpeg_amplify = False #@param {type:"boolean"}

#@markdown ---
#@markdown ## AI translation *(used when the output above is "English (AI translate)")*
translation_provider = "local" #@param ["local", "ollama", "custom", "deepseek", "openrouter", "gemini", "claude", "gpt", "glm", "groq"]
translation_target = "english" #@param ["english", "chinese", "korean", "indonesian", "thai", "spanish", "portuguese", "french", "italian"]
translation_tone = "standard" #@param ["standard", "contextual", "pornify"]
#@markdown <font size="1">local: runs a GGUF model on this session's GPU, no key needed. ollama/custom: an
#@markdown OpenAI-compatible server you give the address of. The rest are cloud services and need a key.</font>
translation_model = "gemma-9b" #@param {type:"string"}
translation_endpoint = "" #@param {type:"string"}
api_key = "" #@param {type:"string"}

#@markdown ---
#@markdown ## Session
opening_credit = "" #@param {type:"string"}
closing_credit = "Subs by WhisperJAV" #@param {type:"string"}
auto_disconnect = True #@param {type:"boolean"}

# ═══════════════════════════════════════════
# CONFIGURATION LOGIC
# ═══════════════════════════════════════════

WHISPERJAV_CMD = "whisperjav"
WHISPERJAV_TRANSLATE_CMD = "whisperjav-translate"

# The release this notebook installs. Kept in one place so Step 3 clones and
# installs the same thing.
WHISPERJAV_RELEASE = "v1.9.5"

# Pipelines that run an OpenAI-Whisper style model and take --model.
WHISPER_PIPELINES = {"balanced", "fidelity", "fast", "faster"}

merge_map = {
    "automatic": "smart_merge",
    "keep all": "full_merge",
    "prefer first pass": "pass1_primary",
    "prefer second pass": "pass2_primary",
    "first pass on overlap": "pass1_overlap",
    "second pass on overlap": "pass2_overlap",
    "longest line": "longest",
}
language_map = {
    "Japanese": "native",
    "English (Whisper direct)": "direct-to-english",
    "English (AI translate)": "llm",
}
transformers_model_map = {
    "kotoba-bilingual-v1.0": "kotoba-tech/kotoba-whisper-bilingual-v1.0",
    "kotoba-v2.0": "kotoba-tech/kotoba-whisper-v2.0",
    "kotoba-v2.1": "kotoba-tech/kotoba-whisper-v2.1",
    "kotoba-v2.2": "kotoba-tech/kotoba-whisper-v2.2",
}

warnings_list = []
errors_list = []


def pick(value):
    """'automatic' means: say nothing on the command line and let WhisperJAV decide."""
    return None if value == "automatic" else value


def ffmpeg_filters(*chosen):
    """
    The effects the user ticked, in the order WhisperJAV applies them, as the
    "ffmpeg-dsp:a,b" form the program accepts. Nothing ticked means loudnorm,
    which is what plain ffmpeg-dsp does anyway.
    """
    names = ["loudnorm", "normalize", "compress", "denoise",
             "highpass", "lowpass", "deess", "amplify"]
    picked = [name for name, ticked in zip(names, chosen) if ticked]
    return ",".join(picked) if picked else "loudnorm"


def describe_pass(n, pipeline, sensitivity, whisper_model, transformers_model,
                  qwen_generator, builtin_vad, segmenter, scene, enhancer):
    """One line telling the user what this pass will actually do."""
    bits = [pipeline, sensitivity]
    if pipeline in WHISPER_PIPELINES and whisper_model != "automatic":
        bits.append(whisper_model)
    elif pipeline == "transformers":
        bits.append(transformers_model)
    elif pipeline == "qwen":
        bits.append(qwen_generator)
    if pipeline == "balanced":
        bits.append("built-in VAD " + ("default" if builtin_vad == "automatic" else builtin_vad))
    elif segmenter != "automatic":
        bits.append(segmenter)
    if scene != "automatic":
        bits.append("scene " + scene)
    if enhancer != "none":
        bits.append("clean-up " + enhancer)
    return " / ".join(bits)


def check_pass(n, pipeline, builtin_vad, segmenter, scene, enhancer):
    """
    Refuse the combinations WhisperJAV itself refuses, here in the form, so the
    user is told before a GPU session is spent rather than after.
    """
    label = "First pass" if n == 1 else "Second pass"
    if pipeline == "balanced" and segmenter != "automatic":
        errors_list.append(
            f"{label}: the balanced pipeline does not take a separate speech segmenter. "
            f"It uses faster-whisper's built-in detector. Either set 'speech segmenter' back to "
            f"'automatic' and choose a build in 'built-in VAD', or pick a different pipeline for "
            f"this pass (fidelity, fast, faster, transformers or qwen all accept '{segmenter}')."
        )
    if pipeline != "balanced" and builtin_vad != "automatic":
        warnings_list.append(
            f"{label}: 'built-in VAD' only applies to the balanced pipeline. It is ignored for "
            f"{pipeline}; use 'speech segmenter' instead."
        )
    if scene == "none" and not use_two_step and pipeline != "qwen":
        errors_list.append(
            f"{label}: scene detection cannot be switched off for a single {pipeline} pass. "
            f"Choose semantic, auditok or silero, or turn two passes on."
        )
    if enhancer != "none" and not use_two_step and pipeline != "qwen":
        warnings_list.append(
            f"{label}: audio clean-up runs only in a two-pass run or with the qwen pipeline. "
            f"It will have no effect on a single {pipeline} pass."
        )


check_pass(1, pass1_pipeline, pass1_builtin_vad, pass1_speech_segmenter,
           pass1_scene_detector, pass1_speech_enhancer)
if use_two_step:
    check_pass(2, pass2_pipeline, pass2_builtin_vad, pass2_speech_segmenter,
               pass2_scene_detector, pass2_speech_enhancer)

if pass1_speech_enhancer in {"clearvoice", "bs-roformer", "zipenhancer"} and \
        pass2_speech_enhancer in {"clearvoice", "bs-roformer", "zipenhancer"} and use_two_step:
    warnings_list.append(
        "Both passes use a GPU audio clean-up. On a T4 that can run out of memory. "
        "Consider ffmpeg-dsp for one of them."
    )

if language_map[subtitle_language] == "llm" and translation_provider not in {"local", "ollama", "custom"} \
        and not api_key:
    errors_list.append(
        f"AI translation with {translation_provider} needs an API key. Either paste one, or choose "
        f"'local' to run the translation on this session's GPU."
    )
if language_map[subtitle_language] == "llm" and translation_provider == "custom" and not translation_endpoint:
    errors_list.append(
        "The 'custom' translation provider needs the address of your OpenAI-compatible server "
        "in 'translation endpoint'."
    )

WHISPERJAV_CONFIG = {
    "release": WHISPERJAV_RELEASE,
    "folder_name": folder_name,
    "subtitle_language": language_map[subtitle_language],
    "use_two_step": use_two_step,
    "merge_strategy": merge_map[merge_method],

    "pass1_pipeline": pass1_pipeline,
    "pass1_sensitivity": pass1_sensitivity,
    "pass1_whisper_model": pick(pass1_whisper_model),
    "pass1_transformers_model": transformers_model_map[pass1_transformers_model],
    "pass1_qwen_generator": pass1_qwen_generator,
    "pass1_builtin_vad": pick(pass1_builtin_vad),
    "pass1_speech_segmenter": pick(pass1_speech_segmenter),
    "pass1_scene_detector": pick(pass1_scene_detector),
    "pass1_speech_enhancer": None if pass1_speech_enhancer == "none" else pass1_speech_enhancer,
    "pass1_ffmpeg_filters": ffmpeg_filters(
        pass1_ffmpeg_loudnorm, pass1_ffmpeg_normalize, pass1_ffmpeg_compress,
        pass1_ffmpeg_denoise, pass1_ffmpeg_highpass, pass1_ffmpeg_lowpass,
        pass1_ffmpeg_deess, pass1_ffmpeg_amplify),

    "pass2_pipeline": pass2_pipeline,
    "pass2_sensitivity": pass2_sensitivity,
    "pass2_whisper_model": pick(pass2_whisper_model),
    "pass2_transformers_model": transformers_model_map[pass2_transformers_model],
    "pass2_qwen_generator": pass2_qwen_generator,
    "pass2_builtin_vad": pick(pass2_builtin_vad),
    "pass2_speech_segmenter": pick(pass2_speech_segmenter),
    "pass2_scene_detector": pick(pass2_scene_detector),
    "pass2_speech_enhancer": None if pass2_speech_enhancer == "none" else pass2_speech_enhancer,
    "pass2_ffmpeg_filters": ffmpeg_filters(
        pass2_ffmpeg_loudnorm, pass2_ffmpeg_normalize, pass2_ffmpeg_compress,
        pass2_ffmpeg_denoise, pass2_ffmpeg_highpass, pass2_ffmpeg_lowpass,
        pass2_ffmpeg_deess, pass2_ffmpeg_amplify),

    "translation_provider": translation_provider,
    "translation_target": translation_target,
    "translation_tone": translation_tone,
    "translation_model": translation_model.strip(),
    "translation_endpoint": translation_endpoint.strip(),
    "api_key": api_key,

    "opening_credit": opening_credit,
    "closing_credit": closing_credit,
    "auto_disconnect": auto_disconnect,

    "whisperjav_cmd": WHISPERJAV_CMD,
    "whisperjav_translate_cmd": WHISPERJAV_TRANSLATE_CMD,

    "_pass1_summary": describe_pass(
        1, pass1_pipeline, pass1_sensitivity, pass1_whisper_model, pass1_transformers_model,
        pass1_qwen_generator, pass1_builtin_vad, pass1_speech_segmenter,
        pass1_scene_detector, pass1_speech_enhancer),
    "_pass2_summary": describe_pass(
        2, pass2_pipeline, pass2_sensitivity, pass2_whisper_model, pass2_transformers_model,
        pass2_qwen_generator, pass2_builtin_vad, pass2_speech_segmenter,
        pass2_scene_detector, pass2_speech_enhancer),
    "_merge_method": merge_method,
}

WHISPERJAV_CONFIG_OK = not errors_list

from IPython.display import display, HTML

for w in warnings_list:
    display(HTML(f'<div style="padding:6px 10px;background:#fef9c3;border-radius:4px;'
                 f'font-size:11px;margin-bottom:4px"><b>Note:</b> {w}</div>'))
for e in errors_list:
    display(HTML(f'<div style="padding:6px 10px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 f'font-size:11px;margin-bottom:4px"><b>Cannot run:</b> {e}</div>'))

if WHISPERJAV_CONFIG_OK:
    lines = [f"<b>Ready</b> &nbsp; WhisperJAV {WHISPERJAV_RELEASE}",
             f"First pass: {WHISPERJAV_CONFIG['_pass1_summary']}"]
    if use_two_step:
        lines.append(f"Second pass: {WHISPERJAV_CONFIG['_pass2_summary']}")
        lines.append(f"Merge: {merge_method}")
    lines.append(f"Subtitles: {subtitle_language}")
    display(HTML('<div style="padding:10px;background:#f0f9ff;border-radius:4px;font-size:11px">'
                 + "<br>".join(lines) + '</div>'))
else:
    display(HTML('<div style="padding:10px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px">Fix the points above and run this cell again before going on.</div>'))

In [ ]:
#@title Step 2: Download from links (optional) { display-mode: "form" }
#@markdown Paste links and they are downloaded, converted to MP3, and saved in the
#@markdown Drive folder, ready for Step 3. Only the MP3 is kept, so a large video
#@markdown becomes a small audio file. No GPU is needed for this step.
links = "" #@param {type:"string"}
#@markdown <font size="1">One link or several, separated by spaces or commas. Streaming
#@markdown pages and direct file links both work. For a long list, keep instead a text
#@markdown file inside the Drive folder &mdash; one link per line, <b>#</b> starts a
#@markdown comment &mdash; and give its path below:</font>
links_file = "links.txt" #@param {type:"string"}
mp3_bitrate = "192k" #@param ["128k", "192k", "256k", "320k"]
allow_playlists = False #@param {type:"boolean"}
#@markdown <font size="1">Members-only or age-gated pages need your browser cookies: export
#@markdown them with a "Get cookies.txt" extension, put the file on Drive and give its path
#@markdown here (e.g. <code>/content/drive/MyDrive/cookies.txt</code>, or just
#@markdown <code>cookies.txt</code> if it sits inside the folder):</font>
cookies_file = "" #@param {type:"string"}

import os, re, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import display, HTML

os.environ['MPLBACKEND'] = 'Agg'


def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")


def section(title):
    print(f"\n{'-'*46}\n{title}\n{'-'*46}")


if 'WHISPERJAV_CONFIG' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 1 first.</b> The folder the files go into is '
                 'chosen there.</div>'))
    raise SystemExit()

folder_name = WHISPERJAV_CONFIG['folder_name']

# ═══════════════════════════════════════════
# CONNECT GOOGLE DRIVE
# ═══════════════════════════════════════════
section("CONNECTING GOOGLE DRIVE")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    folder_path = Path(f"/content/drive/MyDrive/{folder_name}")
    folder_path.mkdir(parents=True, exist_ok=True)
    status(f"Connected: {folder_path}")
except Exception as e:
    status(f"Could not connect to Google Drive: {e}", False)
    raise SystemExit("Google Drive connection failed")


def resolve_in_folder(name):
    """A bare name sits inside the Drive folder; an absolute path is used as given."""
    p = Path(name)
    return p if p.is_absolute() else folder_path / p


# ═══════════════════════════════════════════
# COLLECT THE LINKS
# ═══════════════════════════════════════════
section("COLLECTING LINKS")

MEDIA_EXTS = ('.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v',
              '.mp3', '.wav', '.flac', '.m4a')

found_links = []

# The box takes several links split by spaces, commas or new lines.
for piece in re.split(r'[\s,]+', links.strip()):
    if piece:
        found_links.append(piece)

# A text file in the Drive folder: one link per line, # starts a comment.
list_path = resolve_in_folder(links_file.strip()) if links_file.strip() else None
if list_path and list_path.exists():
    file_links = 0
    for line in list_path.read_text(encoding='utf-8', errors='replace').splitlines():
        line = line.split('#', 1)[0].strip()
        if line:
            found_links.append(line)
            file_links += 1
    status(f"Read {file_links} link(s) from {list_path.name}")
elif list_path:
    print(f"    No {list_path.name} in the folder; using only the links pasted above")

# Keep the first copy of each link, in the order given.
found_links = list(dict.fromkeys(found_links))
if not found_links:
    display(HTML(f'<div style="padding:8px;background:#fef9c3;border-radius:4px;'
                 f'font-size:11px"><b>No links given.</b> Paste at least one link in the box, '
                 f'or save a list in Google Drive/{folder_name}/links.txt.</div>'))
    raise SystemExit()
status(f"{len(found_links)} link(s) queued")

# ═══════════════════════════════════════════
# INSTALL THE DOWNLOADER
# ═══════════════════════════════════════════
section("INSTALLING yt-dlp")

result = subprocess.run("pip install -U -q yt-dlp yt-dlp-ejs", shell=True,
                        capture_output=True, text=True)
if result.returncode != 0:
    # yt-dlp-ejs adds the JavaScript runtime YouTube extraction needs; without
    # it the thousands of other sites still work.
    result = subprocess.run("pip install -U -q yt-dlp", shell=True,
                            capture_output=True, text=True)
if result.returncode != 0:
    status(f"Could not install yt-dlp: {result.stderr.strip()[-300:]}", False)
    raise SystemExit("yt-dlp install failed")

YTDLP = ["yt-dlp"] if shutil.which("yt-dlp") else [sys.executable, "-m", "yt_dlp"]
version = subprocess.run(YTDLP + ["--version"], capture_output=True, text=True)
status(f"yt-dlp {version.stdout.strip()}")

ffmpeg_ok = subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode == 0
if not ffmpeg_ok:
    result = subprocess.run("apt-get install -y -qq ffmpeg", shell=True,
                            capture_output=True, text=True)
    ffmpeg_ok = result.returncode == 0
if not ffmpeg_ok:
    status("ffmpeg is missing and could not be installed", False)
    raise SystemExit("ffmpeg is required for the MP3 conversion")
status("ffmpeg ready")

if cookies_file.strip():
    cookies_path = resolve_in_folder(cookies_file.strip())
    if cookies_path.exists():
        cookies_file = str(cookies_path)
        status(f"Cookies: {cookies_path}")
    else:
        status(f"Cookies file not found: {cookies_path} (continuing without it)", False)
        cookies_file = ""

# ═══════════════════════════════════════════
# DOWNLOAD AND CONVERT
# ═══════════════════════════════════════════
section("DOWNLOADING")

# Download onto the local disk: that is far quicker than writing straight into
# Drive, and only the finished MP3 is moved over. The archive file remembers
# what was fetched, so re-running this step never downloads the same link twice.
tmp_path = Path("/content/whisperjav_downloads")
tmp_path.mkdir(exist_ok=True)
archive_path = folder_path / ".download-archive.txt"

saved, failed, skipped = [], [], 0


def move_to_folder(mp3):
    """Move one finished MP3 into the Drive folder, never overwriting a namesake."""
    target = folder_path / mp3.name
    if target.exists():
        target = folder_path / f"{mp3.stem} ({int(time.time())}){mp3.suffix}"
    shutil.move(str(mp3), target)
    saved.append(target)
    status(f"Saved: {target.name} ({target.stat().st_size / 1e6:.1f} MB)")


# Anything left in the download folder by an interrupted earlier run is still
# good; rescue it before starting anew.
for leftover in sorted(tmp_path.glob('*.mp3')):
    move_to_folder(leftover)

for i, url in enumerate(found_links, 1):
    print(f"\n[{i}/{len(found_links)}] {url}")
    before = set(tmp_path.glob('*.mp3'))

    cmd = YTDLP + [
        "-x", "--audio-format", "mp3", "--audio-quality", mp3_bitrate,
        "--windows-filenames", "--trim-filenames", "150",
        "--download-archive", str(archive_path),
        "--yes-playlist" if allow_playlists else "--no-playlist",
        "--retries", "3",
        "-P", str(tmp_path), "-o", "%(title)s.%(ext)s",
    ]
    if cookies_file:
        cmd += ["--cookies", cookies_file]
    cmd.append(url)

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               bufsize=1, universal_newlines=True)
    output_lines = []
    for line in process.stdout:
        print(f"    {line}", end='')
        output_lines.append(line)
    process.wait()

    new_mp3s = set(tmp_path.glob('*.mp3')) - before

    if process.returncode != 0 and url.split('?')[0].lower().endswith(MEDIA_EXTS):
        # yt-dlp does not know the site, but the link points at a file plain and
        # simple: fetch it and turn the sound into MP3 with ffmpeg.
        name = re.sub(r'[^\w.\-]+', '_', Path(url.split('?')[0]).name) or f"link{i}"
        fetched = tmp_path / name[:150]
        if not fetched.suffix:
            fetched = fetched.with_suffix('.media')
        print("    Unknown site for yt-dlp; trying a plain file download...")
        r = subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(fetched), url],
                           capture_output=True, text=True)
        if r.returncode == 0 and fetched.exists() and fetched.stat().st_size > 0:
            if fetched.suffix.lower() == '.mp3':
                new_mp3s = {fetched}
            else:
                mp3_dest = fetched.with_suffix('.mp3')
                r = subprocess.run(["ffmpeg", "-y", "-i", str(fetched), "-vn",
                                    "-codec:a", "libmp3lame", "-b:a", mp3_bitrate,
                                    str(mp3_dest)], capture_output=True, text=True)
                fetched.unlink(missing_ok=True)
                if r.returncode == 0 and mp3_dest.exists():
                    new_mp3s = {mp3_dest}

    if new_mp3s:
        for mp3 in sorted(new_mp3s):
            move_to_folder(mp3)
    elif 'recorded in the archive' in ''.join(output_lines):
        skipped += 1
        print("    Already fetched on an earlier run; skipping.")
    else:
        failed.append(url)
        status("Could not download this link", False)

# ═══════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════
section("DOWNLOADS COMPLETE")
print(f"Saved: {len(saved)}   Already had: {skipped}   Failed: {len(failed)}")
if failed:
    print("Failed links:")
    for url in failed:
        print(f"  - {url}")
if not saved and (failed or skipped):
    box_color, border, word = "#fef9c3", "#ca8a04", "Nothing new"
else:
    box_color, border, word = "#f0fdf4", "#16a34a", "Done"
message = (f"<b>{word}:</b> {len(saved)} new MP3 file(s) in Google Drive/{folder_name}/, "
           f"{len(failed)} link(s) failed.")
display(HTML(f'<div style="padding:8px 10px;background:{box_color};border-radius:4px;'
             f'border-left:2px solid {border};font-size:11px">{message} '
             f'Continue with <b>Step 3</b> to transcribe.</div>'))


In [3]:
#@title Step 3: Transcribe { display-mode: "form" }
#@markdown Connect Drive, install WhisperJAV, transcribe every media file in the folder —
#@markdown the ones you put there and the ones Step 2 downloaded — and add credits.

import os, sys, subprocess, shlex, time
os.environ['TORCH_HUB_TRUST_REPO'] = '1'
from pathlib import Path
from IPython.display import display, HTML

# Colab sets MPLBACKEND to 'matplotlib_inline', which is not available to the
# subprocesses we start. Use the non-interactive backend instead.
os.environ['MPLBACKEND'] = 'Agg'

WHISPER_PIPELINES = {"balanced", "fidelity", "fast", "faster"}


def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")


def section(title):
    print(f"\n{'-'*46}\n{title}\n{'-'*46}")


if 'WHISPERJAV_CONFIG' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 1 first.</b></div>'))
    raise SystemExit()
if not globals().get('WHISPERJAV_CONFIG_OK', False):
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Step 1 reported a setting that cannot run.</b> '
                 'Fix it and run Step 1 again.</div>'))
    raise SystemExit()

cfg = WHISPERJAV_CONFIG

# ═══════════════════════════════════════════
# PRE-FLIGHT CHECKS
# ═══════════════════════════════════════════
section("PRE-FLIGHT CHECKS")

# WhisperJAV supports Python 3.10 to 3.13. Colab's runtime Python is not ours to
# change, so this only reports; it never tries to swap the interpreter.
py = sys.version_info
print(f"Python: {sys.version.split()[0]}")
if py < (3, 10) or py >= (3, 14):
    display(HTML(
        '<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;font-size:11px">'
        f'<b>Python {sys.version.split()[0]} is not supported.</b> WhisperJAV needs Python 3.10 to 3.13. '
        'This is the Python that Colab gave this session; try a different Colab runtime.</div>'))
    raise SystemExit(f"Python {sys.version.split()[0]} not supported")
status("Python version supported (3.10-3.13)")

section("CONNECTING GOOGLE DRIVE")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    folder_path = Path(f"/content/drive/MyDrive/{cfg['folder_name']}")
    folder_path.mkdir(parents=True, exist_ok=True)
    status(f"Connected: {folder_path}")
except Exception as e:
    status(f"Could not connect to Google Drive: {e}", False)
    raise SystemExit("Google Drive connection failed")

section("CHECKING GPU")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    status("No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.", False)
    raise SystemExit("No GPU detected")
status(f"GPU: {gpu_check.stdout.strip()}")

# ═══════════════════════════════════════════
# INSTALL WHISPERJAV
# ═══════════════════════════════════════════
section(f"INSTALLING WHISPERJAV {cfg['release']}")
install_start = time.time()

REPO_URL = "https://github.com/meizhong986/WhisperJAV.git"
REPO_PATH = "/content/WhisperJAV"
SCRIPT_PATH = f"{REPO_PATH}/installer/install_colab.sh"
RELEASE = cfg['release']


def run_installer():
    env = {**os.environ,
           "PATH": f"{os.environ.get('PATH', '')}:{os.path.expanduser('~/.local/bin')}",
           # Install the same release the notebook was cloned at, not the tip of main.
           "WHISPERJAV_REF": RELEASE}
    process = subprocess.Popen(["bash", SCRIPT_PATH], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=1, text=True, env=env)
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
    process.wait()
    return process.returncode


check = subprocess.run(["python3", "-c", "import whisperjav"], capture_output=True)
if check.returncode == 0:
    status("WhisperJAV already installed in this session (skipping)")
else:
    print(f"Installing WhisperJAV {RELEASE}...\n")
    sys.stdout.flush()
    if not os.path.exists(REPO_PATH):
        print(f"Cloning {REPO_URL} at {RELEASE}...")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", RELEASE, REPO_URL, REPO_PATH],
            capture_output=True, text=True)
        if result.returncode != 0:
            status(f"Could not fetch WhisperJAV {RELEASE}: {result.stderr.strip()}", False)
            raise SystemExit("Clone failed")
    if not os.path.exists(SCRIPT_PATH):
        status(f"Install script missing: {SCRIPT_PATH}", False)
        raise SystemExit("Install script missing from repo")
    if run_installer() != 0:
        status("Installation failed", False)
        raise SystemExit("Installation failed")

# Audio clean-up backends that are not part of the default install.
extra_packages = set()
for enhancer in (cfg['pass1_speech_enhancer'], cfg['pass2_speech_enhancer'] if cfg['use_two_step'] else None):
    if enhancer == 'clearvoice':
        extra_packages.add('clearvoice')
    elif enhancer == 'bs-roformer':
        extra_packages.add('bs-roformer-infer')
    elif enhancer == 'htdemucs':
        extra_packages.add('demucs')
if extra_packages:
    pkg_list = ' '.join(sorted(extra_packages))
    uv_path = os.path.expanduser("~/.local/bin/uv")
    if os.path.exists(uv_path):
        result = subprocess.run(f"{uv_path} pip install --system {pkg_list}", shell=True,
                                capture_output=True, text=True)
    else:
        result = subprocess.run(f"pip install -q {pkg_list}", shell=True,
                                capture_output=True, text=True)
    status(f"Audio clean-up packages ({pkg_list})", result.returncode == 0)

status(f"Installation finished ({time.time()-install_start:.0f}s)")

# Colab moved to CUDA 13 (October 2026; #447, #456). faster-whisper's engine, CTranslate2, is
# built for CUDA 12 and needs libcublas.so.12; the local-translation build of llama-cpp-python
# needs the CUDA 12 runtime too. So, when Colab's PyTorch is a CUDA 13 build, add those CUDA 12
# libraries beside it and register their folders with the system loader. PyTorch is left alone.
#  - Installed with the same Python that runs `whisperjav`, so the libraries land where it looks.
#  - Registered in /etc/ld.so.conf.d rather than an environment variable, so they stay found for
#    every WhisperJAV process and after Runtime -> Restart session.
#  - cuDNN is NOT added: nvidia-cudnn-cu12 writes the same files into the same folder
#    (nvidia/cudnn/lib) as PyTorch's nvidia-cudnn-cu13 and would replace PyTorch's copy.
section("CHECKING CUDA VERSION")
CUDA12_PACKAGES = ["nvidia-cublas-cu12==12.9.2.10", "nvidia-cuda-runtime-cu12==12.9.79"]
CUDA12_LIBS = {"cublas": "libcublas.so.12", "cuda_runtime": "libcudart.so.12"}
LD_CONF = "/etc/ld.so.conf.d/whisperjav-cuda12.conf"


def whisperjav_python():
    """The interpreter named on the first line of the installed `whisperjav` command."""
    import shutil
    exe = shutil.which(cfg['whisperjav_cmd'])
    if exe:
        try:
            with open(exe, encoding='utf-8', errors='replace') as f:
                first = f.readline().strip()
            if first.startswith('#!'):
                interpreter = first[2:].strip().split()[0]
                if os.path.exists(interpreter):
                    return interpreter
        except OSError:
            pass
    return "python3"


def run_py(code):
    r = subprocess.run([WJ_PYTHON, "-c", code], capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()


WJ_PYTHON = whisperjav_python()
print(f"WhisperJAV runs on: {WJ_PYTHON}")
_, torch_cuda, _ = run_py("import torch; print(torch.version.cuda or '')")

if torch_cuda.startswith("13"):
    print(f"Colab's PyTorch is built for CUDA {torch_cuda}; faster-whisper needs CUDA 12 libraries.")
    result = subprocess.run([WJ_PYTHON, "-m", "pip", "install", "-q", *CUDA12_PACKAGES],
                            capture_output=True, text=True)
    if result.returncode != 0:
        status("Could not install the CUDA 12 libraries", False)
        print(result.stderr.strip()[-2000:])
        raise SystemExit("CUDA 12 library install failed")

    folders = []
    for module, lib in CUDA12_LIBS.items():
        rc, out, err = run_py(f"import os, nvidia.{module} as m; print(os.path.join(list(m.__path__)[0], 'lib'))")
        if rc != 0 or not os.path.exists(os.path.join(out, lib)):
            status(f"{lib} not found after install", False)
            print(err[-1000:])
            raise SystemExit("CUDA 12 library missing")
        folders.append(out)
    with open(LD_CONF, "w") as f:
        f.write("\n".join(folders) + "\n")
    subprocess.run(["ldconfig"], capture_output=True, text=True)

    # Load the libraries the way a WhisperJAV process will: PyTorch first, then by name.
    check_code = ("import ctypes, torch\n"
                  "print('torch-cuda', torch.version.cuda)\n"
                  "for name in ('libcublas.so.12', 'libcudart.so.12', 'libcudnn.so.9'):\n"
                  "    try:\n"
                  "        ctypes.CDLL(name); print('ok', name)\n"
                  "    except OSError as e:\n"
                  "        print('missing', name, e)\n")
    rc, out, err = run_py(check_code)
    print(out or err[-1000:])
    if "missing libcublas.so.12" in out or "missing libcudart.so.12" in out or rc != 0:
        status("The CUDA 12 libraries are installed but cannot be loaded", False)
        raise SystemExit("CUDA 12 libraries not loadable")
    if "missing libcudnn.so.9" in out:
        status("cuDNN 9 could not be loaded by name; balanced/faster/fast may fail", False)
    if not out.startswith(f"torch-cuda {torch_cuda}"):
        status("PyTorch's CUDA version changed; this should not happen", False)
        raise SystemExit("PyTorch changed")
    status(f"CUDA 12 libraries added; PyTorch unchanged (CUDA {torch_cuda})")
elif torch_cuda:
    status(f"PyTorch CUDA {torch_cuda}: no change needed")
else:
    status("Could not read PyTorch's CUDA version; continuing", False)

# ═══════════════════════════════════════════
# FIND MEDIA FILES
# ═══════════════════════════════════════════
section("SCANNING FILES")
media_types = {'.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v',
               '.mp3', '.wav', '.flac', '.m4a'}
videos = [f for f in folder_path.iterdir() if f.suffix.lower() in media_types]
if not videos:
    status(f"No media files in Google Drive/{cfg['folder_name']}/ — put files there "
           f"yourself, or paste links in Step 2", False)
    raise SystemExit("No media files found")
status(f"Found {len(videos)} file(s)")
for v in videos[:5]:
    print(f"  - {v.name}")
if len(videos) > 5:
    print(f"  ... and {len(videos)-5} more")

existing_srts = set(folder_path.glob('*.srt'))

# ═══════════════════════════════════════════
# BUILD THE COMMAND
# ═══════════════════════════════════════════
section("TRANSCRIBING")

cmd = [cfg['whisperjav_cmd'], str(folder_path), '--output-dir', str(folder_path)]


def single_pass_flags(cmd):
    """Flags for a one-pass run. Each pipeline family has its own flag names."""
    p = cfg['pass1_pipeline']
    cmd += ['--mode', p]

    if p == 'qwen':
        cmd += ['--qwen-sensitivity', cfg['pass1_sensitivity']]
        cmd += ['--qwen-generator', cfg['pass1_qwen_generator']]
        if cfg['pass1_speech_segmenter']:
            cmd += ['--qwen-segmenter', cfg['pass1_speech_segmenter']]
        if cfg['pass1_scene_detector']:
            cmd += ['--qwen-scene', cfg['pass1_scene_detector']]
        if cfg['pass1_speech_enhancer']:
            cmd += ['--qwen-enhancer', cfg['pass1_speech_enhancer']]
        return

    cmd += ['--sensitivity', cfg['pass1_sensitivity']]

    if p == 'transformers':
        cmd += ['--hf-model-id', cfg['pass1_transformers_model']]
        scene = cfg['pass1_scene_detector']
        if scene in ('auditok', 'silero', 'none'):
            cmd += ['--hf-scene', scene]
        elif scene == 'semantic':
            print("Note: the transformers pipeline has no semantic scene detection; "
                  "leaving scene detection at its default.")
    else:
        if cfg['pass1_whisper_model']:
            cmd += ['--model', cfg['pass1_whisper_model']]
        if cfg['pass1_scene_detector'] and cfg['pass1_scene_detector'] != 'none':
            cmd += ['--scene-detection-method', cfg['pass1_scene_detector']]

    # Balanced runs faster-whisper's built-in detector and refuses a separate
    # segmenter; every other pipeline is the other way round.
    if p == 'balanced':
        if cfg['pass1_builtin_vad']:
            cmd += ['--vad-version', cfg['pass1_builtin_vad']]
    elif cfg['pass1_speech_segmenter']:
        cmd += ['--speech-segmenter', cfg['pass1_speech_segmenter']]


def two_pass_flags(cmd):
    """Flags for a two-pass run. Here every pipeline uses the same --passN- names."""
    cmd += ['--ensemble', '--merge-strategy', cfg['merge_strategy']]
    for n in (1, 2):
        p = cfg[f'pass{n}_pipeline']
        pre = f'--pass{n}-'
        cmd += [pre + 'pipeline', p, pre + 'sensitivity', cfg[f'pass{n}_sensitivity']]

        if p == 'qwen':
            import json as _json
            cmd += [pre + 'qwen-params',
                    _json.dumps({"generator_backend": cfg[f'pass{n}_qwen_generator']})]
        elif p == 'transformers':
            cmd += [pre + 'model', cfg[f'pass{n}_transformers_model']]
        elif cfg[f'pass{n}_whisper_model']:
            cmd += [pre + 'model', cfg[f'pass{n}_whisper_model']]

        if p == 'balanced':
            if cfg[f'pass{n}_builtin_vad']:
                cmd += [pre + 'vad-version', cfg[f'pass{n}_builtin_vad']]
        elif cfg[f'pass{n}_speech_segmenter']:
            cmd += [pre + 'speech-segmenter', cfg[f'pass{n}_speech_segmenter']]

        if cfg[f'pass{n}_scene_detector']:
            cmd += [pre + 'scene-detector', cfg[f'pass{n}_scene_detector']]

        enhancer = cfg[f'pass{n}_speech_enhancer']
        if enhancer == 'ffmpeg-dsp':
            # v1.9.3: the program takes the chosen filters after a colon.
            cmd += [pre + 'speech-enhancer', f"ffmpeg-dsp:{cfg[f'pass{n}_ffmpeg_filters']}"]
        elif enhancer:
            cmd += [pre + 'speech-enhancer', enhancer]


if cfg['use_two_step']:
    two_pass_flags(cmd)
    print("Two passes")
    print(f"  First pass:  {cfg['_pass1_summary']}")
    print(f"  Second pass: {cfg['_pass2_summary']}")
    print(f"  Merge:       {cfg['_merge_method']}")
else:
    single_pass_flags(cmd)
    print(f"One pass: {cfg['_pass1_summary']}")

if cfg['subtitle_language'] == 'direct-to-english':
    cmd += ['--subs-language', 'direct-to-english']
    print("Subtitles: English, translated by Whisper itself")
else:
    cmd += ['--subs-language', 'native']
    if cfg['subtitle_language'] == 'llm':
        print("Subtitles: Japanese now; Step 4 translates them")
    else:
        print("Subtitles: Japanese")

print(f"Input: {folder_path}\n")

full_cmd = shlex.join(cmd)
print(f"Command: {full_cmd}\n")
process = subprocess.Popen(full_cmd, shell=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    status("Transcription failed", False)
    raise SystemExit("Transcription failed")

all_srts = set(folder_path.glob('*.srt'))
new_srts = [f for f in (all_srts - existing_srts)
            if '.pass1.' not in f.name and '.pass2.' not in f.name]
new_srts.sort(key=lambda x: x.name)

WHISPERJAV_NEW_SRTS = new_srts
WHISPERJAV_FOLDER_PATH = folder_path

status(f"Created {len(new_srts)} subtitle file(s)")

# ═══════════════════════════════════════════
# ADD CREDITS
# ═══════════════════════════════════════════
section("ADDING CREDITS")
if cfg['opening_credit'] or cfg['closing_credit']:
    credits_count = 0
    for srt_file in new_srts:
        try:
            content = srt_file.read_text(encoding='utf-8')
            if cfg['opening_credit']:
                content = f"0\n00:00:00,000 --> 00:00:00,500\n{cfg['opening_credit']}\n\n" + content
            if cfg['closing_credit']:
                content += f"\n9999\n23:59:58,000 --> 23:59:59,000\n{cfg['closing_credit']}\n"
            srt_file.write_text(content, encoding='utf-8')
            credits_count += 1
        except Exception as e:
            print(f"  Could not add credits to {srt_file.name}: {e}")
    status(f"Credits added to {credits_count} file(s)")
else:
    status("No credits set")

try:
    if 'process' in locals():
        process.terminate()
        del process
    subprocess.run(["pkill", "-f", "whisperjav"], capture_output=True)
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

section("TRANSCRIPTION COMPLETE")
if cfg['subtitle_language'] == 'llm':
    display(HTML(f'<div style="padding:8px 10px;background:#fef9c3;border-radius:4px;'
                 f'border-left:2px solid #ca8a04;font-size:11px"><b>Done:</b> {len(new_srts)} '
                 f'Japanese subtitle file(s). Run Step 4 to translate them.</div>'))
else:
    display(HTML(f'<div style="padding:8px 10px;background:#f0fdf4;border-radius:4px;'
                 f'border-left:2px solid #16a34a;font-size:11px"><b>Done:</b> {len(new_srts)} '
                 f'subtitle file(s) in Google Drive/{cfg["folder_name"]}/</div>'))
    if cfg['auto_disconnect']:
        print("\nDisconnecting in 10 seconds to save GPU credits...")
        time.sleep(10)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            pass


----------------------------------------------
PRE-FLIGHT CHECKS
----------------------------------------------
Python: 3.13.16
OK   Python version supported (3.10-3.13)

----------------------------------------------
CONNECTING GOOGLE DRIVE
----------------------------------------------
Mounted at /content/drive
OK   Connected: /content/drive/MyDrive/WhisperJAV

----------------------------------------------
CHECKING GPU
----------------------------------------------
OK   GPU: Tesla T4, 15360 MiB

----------------------------------------------
INSTALLING WHISPERJAV v1.9.5
----------------------------------------------
Installing WhisperJAV v1.9.5...

Cloning https://github.com/meizhong986/WhisperJAV.git at v1.9.5...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WhisperJAV Colab Installer
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[INFO] Detected Google Colab environment
[INFO] GPU: Tesla T4
[INFO] Driver: 580.82.07
[INFO] Python: 3.13.16
[INFO] PyTorch: 2.11.0+cu130 (

In [ ]:
#@title Step 4: AI Translation { display-mode: "form" }
#@markdown Translates the subtitle files Step 3 produced. Runs only when the output in Step 1
#@markdown is **English (AI translate)**.

import os, shlex, subprocess, time
from pathlib import Path
from IPython.display import display, HTML

os.environ['MPLBACKEND'] = 'Agg'


def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")


def section(title):
    print(f"\n{'-'*46}\n{title}\n{'-'*46}")


# Free the GPU before the translation model is loaded.
try:
    subprocess.run(["pkill", "-f", "whisperjav"], capture_output=True)
    subprocess.run(["pkill", "-f", "llama_cpp"], capture_output=True)
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

if 'WHISPERJAV_CONFIG' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 1 first.</b></div>'))
    raise SystemExit()
if 'WHISPERJAV_NEW_SRTS' not in dir() or 'WHISPERJAV_FOLDER_PATH' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px\"><b>Run Step 3 first.</b></div>'))
    raise SystemExit()

cfg = WHISPERJAV_CONFIG
new_srts = WHISPERJAV_NEW_SRTS
folder_path = WHISPERJAV_FOLDER_PATH

if cfg['subtitle_language'] != 'llm':
    display(HTML('<div style="padding:8px 10px;background:#f0f9ff;border-radius:4px;'
                 'border-left:2px solid #3b82f6;font-size:11px"><b>Skipped:</b> the output chosen in '
                 'Step 1 is not "English (AI translate)".</div>'))
    raise SystemExit()

provider = cfg['translation_provider']
runs_here = provider in ('local', 'ollama', 'custom')

if not runs_here and not cfg['api_key']:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>No API key.</b> Paste one in Step 1, or choose the "local" '
                 'provider to translate on this session\'s GPU.</div>'))
    raise SystemExit()

if not new_srts:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Nothing to translate.</b></div>'))
    raise SystemExit()

if not runs_here:
    env_map = {
        "deepseek": "DEEPSEEK_API_KEY",
        "openrouter": "OPENROUTER_API_KEY",
        "gemini": "GEMINI_API_KEY",
        "claude": "ANTHROPIC_API_KEY",
        "gpt": "OPENAI_API_KEY",
        "glm": "GLM_API_KEY",
        "groq": "GROQ_API_KEY",
    }
    os.environ[env_map.get(provider, "API_KEY")] = cfg['api_key']

section("AI TRANSLATION")
print(f"Provider: {provider}")
if provider == 'local':
    print(f"Model: {cfg['translation_model'] or 'gemma-9b'} (downloaded on first use, about 5 GB)")
elif cfg['translation_model']:
    print(f"Model: {cfg['translation_model']}")
if cfg['translation_endpoint']:
    print(f"Server: {cfg['translation_endpoint']}")
print(f"Target language: {cfg['translation_target']}")
print(f"Tone: {cfg['translation_tone']}")
print(f"Files: {len(new_srts)}\n")

translated_files = []
failed_files = []

for i, srt_file in enumerate(new_srts, 1):
    print(f"[{i}/{len(new_srts)}] {srt_file.name}")

    translate_cmd = [
        cfg['whisperjav_translate_cmd'],
        '-i', str(srt_file),
        '--provider', provider,
        '-t', cfg['translation_target'],
        '--tone', cfg['translation_tone'],
        '--stream',
    ]
    if cfg['translation_model']:
        translate_cmd += ['--model', cfg['translation_model']]
    if cfg['translation_endpoint']:
        translate_cmd += ['--endpoint', cfg['translation_endpoint']]

    full_cmd = shlex.join(translate_cmd)

    try:
        process = subprocess.Popen(full_cmd, shell=True, stdout=subprocess.PIPE,
                                   stderr=subprocess.PIPE, bufsize=1, universal_newlines=True)
        for line in process.stderr:
            print(f"    {line}", end='')
        stdout_output, _ = process.communicate()

        if process.returncode == 0:
            output_path = stdout_output.strip()
            if output_path:
                translated_files.append(Path(output_path))
            status(f"Translated {srt_file.name}")
        else:
            status(f"Could not translate {srt_file.name}", False)
            failed_files.append(srt_file)
    except Exception as e:
        status(f"Could not translate {srt_file.name}: {e}", False)
        failed_files.append(srt_file)

    print()

section("COMPLETE")
if failed_files:
    display(HTML(f'<div style="padding:8px 10px;background:#fef9c3;border-radius:4px;'
                 f'border-left:2px solid #ca8a04;font-size:11px"><b>Partly done:</b> '
                 f'{len(translated_files)} of {len(new_srts)} translated, {len(failed_files)} failed. '
                 f'The Japanese subtitles are still in Google Drive/{cfg["folder_name"]}/.</div>'))
else:
    display(HTML(f'<div style="padding:8px 10px;background:#f0fdf4;border-radius:4px;'
                 f'border-left:2px solid #16a34a;font-size:11px"><b>Done:</b> {len(new_srts)} Japanese '
                 f'and {len(translated_files)} translated subtitle file(s) in '
                 f'Google Drive/{cfg["folder_name"]}/.</div>'))

if cfg.get('auto_disconnect'):
    print("\nDisconnecting in 10 seconds to save GPU credits...")
    time.sleep(10)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass
else:
    print("\nRemember to disconnect the runtime to save GPU credits.")


----------------------------------------------
AI TRANSLATION
----------------------------------------------
Provider: local
Model: gemma-9b (downloaded on first use, about 5 GB)
Target language: english
Tone: standard
Files: 1

[1/1] DANDYA-045.ja.whisperjav.srt
    
      WHISPERJAV TRANSLATE - CLI Configuration
      Files to process: 1
      Provider: local
      Model: gemma-9b
      Source: japanese -> Target: english
      Tone: standard
      Scene threshold: 60.0s
      Max batch size: 30
      Instructions: /tmp/whisperjav_translate/instructions_standard.txt
      Provider options: {'temperature': 0.5, 'top_p': 0.9, 'max_retries': 3, 'backoff_time': 5.0}
      Debug: False
    
      [DEPRECATION WARNING] --provider local is deprecated as of v1.8.10.
      Please migrate to --provider ollama for better stability and model support.
      The local LLM server (llama-cpp-python) will be removed in a later release.
    
      NOTE: Local LLM provider - server will be started
   


Disconnecting in 10 seconds to save GPU credits...
